# 🎞️ ANIMA on Diffucore UI — Google Colab (free tier)

Run **ANIMA** (CircleStone Labs' Cosmos-Predict2 2B text-to-image DiT) through
**[Diffucore UI](https://github.com/nawka12/diffucore-ui)** — a clean, linear
workspace (no ComfyUI nodes), Apache-2.0, powered by the
**[Diffucore](https://github.com/nawka12/diffucore)** inference engine.

**Run the cells in order, top to bottom:**

1. **Config & Google Drive** — mounts Drive, creates the persistent folders
2. **Install** — clones Diffucore UI + engine, installs deps (~3–5 min, every session)
3. **Download Anima** — pulls the DiT / text encoder / VAE into Drive (first run only, ~5 GB)
4. **Civitai / HF importer** — paste download URLs to add LoRAs or new checkpoints, **anytime, even mid-session**
5. **Launch** — starts the UI and prints your public **ngrok** URL (works from your phone)
6. **Status / stop** — utilities

**Persistence model (Google Drive):**
- `MyDrive/AnimaDiffucore/models/…` — checkpoints, DiTs, VAEs, text encoders, **LoRAs**, detailers (survive sessions)
- `MyDrive/AnimaDiffucore/outputs/YYYY-MM-DD/…` — every generated PNG with full metadata (survives sessions)
- Engine/UI code and pip installs are ephemeral and re-run each session (that's the Colab way)

**Requirements:** a free [ngrok](https://dashboard.ngrok.com/get-started/your-authtoken) authtoken (put it in the launch cell or in Colab Secrets as `NGROK_TOKEN`), and optionally a [civitai.com](https://civitai.com) API token for gated models.

> 💡 On a free **T4** expect roughly **45–90 s per image** at 832×1216 / 1024². The first image pays a one-time GPU warmup.
>
> ⚠️ The ngrok URL is **public** — anyone who has the link can use your session. Set a password in the launch cell to gate it.

### 1️⃣ Config & Google Drive

In [ ]:
# @title 1️⃣ Config & Google Drive {"form-width":"30%"}

USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_FOLDER = "AnimaDiffucore"  # @param {type:"string"}

import os
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        print("Mounting Google Drive ...")
        drive.mount("/content/drive")
    GDRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
else:
    GDRIVE = Path("/content/anima_local")  # ephemeral, wiped when the VM resets

# These two persist across sessions: models/ holds checkpoints + LoRAs,
# outputs/ is where the UI saves every generated PNG automatically.
DRIVE_MODELS = GDRIVE / "models"
DRIVE_OUTPUTS = GDRIVE / "outputs"
for sub in ("checkpoints", "diffusion-models", "vae", "text-encoders",
            "loras", "detailers", "upscalers", "oss_cache", "teacache_cache"):
    (DRIVE_MODELS / sub).mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

UI_DIR = Path("/content/diffucore-ui")   # engine + UI code (re-cloned each session)
PORT = 7860

print("Drive root   :", GDRIVE)
print("Models folder:", DRIVE_MODELS)
print("Outputs      :", DRIVE_OUTPUTS)

### 2️⃣ Install Diffucore UI + engine

In [ ]:
# @title 2️⃣ Install Diffucore UI + engine (~3–5 min, run every session) {"form-width":"30%"}

FORCE_REINSTALL = False  # @param {type:"boolean"}

import subprocess, sys
from pathlib import Path

if FORCE_REINSTALL or not (UI_DIR / "backend" / "app.py").exists():
    !rm -rf $UI_DIR
    !git clone --recurse-submodules https://github.com/nawka12/diffucore-ui.git $UI_DIR
else:
    !git -C $UI_DIR pull --ff-only || true
!git -C $UI_DIR submodule update --init --recursive || true

# Torch: Colab already ships a CUDA build — repair it only if broken.
probe = subprocess.run(
    [sys.executable, "-c",
     "import torch; print('CUDA_OK' if torch.cuda.is_available() else 'CUDA_BAD')"],
    capture_output=True, text=True)
if "CUDA_OK" in probe.stdout:
    pass
elif "ModuleNotFoundError" in (probe.stdout + probe.stderr):
    print("Installing the CUDA build of torch (cu124) ...")
    !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124
else:
    print("⚠️ torch found but CUDA is unavailable — switch the runtime to GPU")
    print("   (Runtime → Change runtime type → T4 GPU) for usable speeds.")

print("Installing Diffucore UI dependencies ...")
!pip install -q -r $UI_DIR/requirements.txt
!pip install -q -e $UI_DIR/diffucore
!pip install -q pyngrok

# Point the app at Google Drive: models/ and outputs/ become symlinks, so every
# downloaded model, LoRA and generated image lands in Drive automatically.
import shutil
for link, target in ((UI_DIR / "models", DRIVE_MODELS), (UI_DIR / "outputs", DRIVE_OUTPUTS)):
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link, ignore_errors=True)
    link.symlink_to(target)

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("Install complete — models/ and outputs/ now live in", GDRIVE)

### 3️⃣ Download the Anima model files (into Drive)

In [ ]:
# @title 3️⃣ Download the Anima model (first run / new DiT — saved to Drive) {"form-width":"30%"}

DOWNLOAD_TURBO_DIT = False    # @param {type:"boolean"}
DOWNLOAD_TURBO_LORA = True    # @param {type:"boolean"}
INSTALL_FACE_DETAILER = True  # @param {type:"boolean"}

import re, time, requests
from pathlib import Path
from tqdm.auto import tqdm

HF_ANIMA = "https://huggingface.co/circlestone-labs/Anima/resolve/main"
HF_ANIMA_LORAS = "https://huggingface.co/circlestone-labs/Anima-Official-LoRAs/resolve/main"
HF_ADETAILER = "https://huggingface.co/Bingsu/adetailer/resolve/main"

def fetch(url: str, dest_dir, filename=None) -> Path:
    """Streaming downloader with resume (.part files). Works for HF and Civitai."""
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    last_err = None
    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=(15, 60), allow_redirects=True) as r:
                r.raise_for_status()
                if filename is None:
                    m = re.search(r'filename="?([^";]+)"?',
                                  r.headers.get("Content-Disposition", ""))
                    filename = m.group(1) if m else url.split("?")[0].rstrip("/").split("/")[-1]
                final = dest_dir / filename
                part = dest_dir / (filename + ".part")
                expected = int(r.headers.get("Content-Length", 0))
                if final.exists() and (expected == 0 or final.stat().st_size == expected):
                    print(f"✓ already present: {filename}")
                    return final
                resume = part.stat().st_size if part.exists() else 0
                if resume:
                    rr = requests.get(url, headers={"Range": f"bytes={resume}-"},
                                      stream=True, timeout=(15, 60), allow_redirects=True)
                    if rr.status_code == 206:
                        r = rr
                        expected = resume + int(rr.headers.get("Content-Length", 0))
                    else:
                        resume = 0
                        part.unlink(missing_ok=True)
                with open(part, "ab" if resume else "wb") as f, tqdm(
                        total=expected or None, initial=resume, unit="B",
                        unit_scale=True, desc=filename) as bar:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
                        bar.update(len(chunk))
            part.rename(final)
            return final
        except Exception as e:
            last_err = e
            print(f"retry {attempt + 1}/3 failed: {e}")
            time.sleep(5)
    raise RuntimeError(f"download failed: {url} ({last_err})")

JOBS = [
    (f"{HF_ANIMA}/split_files/diffusion_models/anima-base-v1.0.safetensors",
     DRIVE_MODELS / "diffusion-models", "anima-base-v1.0.safetensors"),
    (f"{HF_ANIMA}/split_files/text_encoders/qwen_3_06b_base.safetensors",
     DRIVE_MODELS / "text-encoders", "qwen_3_06b_base.safetensors"),
    (f"{HF_ANIMA}/split_files/vae/qwen_image_vae.safetensors",
     DRIVE_MODELS / "vae", "qwen_image_vae.safetensors"),
]
if DOWNLOAD_TURBO_DIT:
    # Official turbo DiT — few steps, CFG 1. (Skip the turbo LoRA if you use this.)
    JOBS.append((f"{HF_ANIMA}/split_files/diffusion_models/anima-turbo-v1.0.safetensors",
                 DRIVE_MODELS / "diffusion-models", "anima-turbo-v1.0.safetensors"))
if DOWNLOAD_TURBO_LORA:
    # Official turbo LoRA for the base DiT — enables fast 8–12 step, CFG 1 generation.
    JOBS.append((f"{HF_ANIMA_LORAS}/anima-turbo-lora-v0.2.safetensors",
                 DRIVE_MODELS / "loras", "anima-turbo-lora-v0.2.safetensors"))
if INSTALL_FACE_DETAILER:
    # YOLO face detector for Diffucore UI's built-in ADetailer-style face fix.
    JOBS.append((f"{HF_ADETAILER}/face_yolov8n.pt",
                 DRIVE_MODELS / "detailers", "face_yolov8n.pt"))

for url, dest, name in JOBS:
    fetch(url, dest, name)

print("\nAll Anima files are saved in Google Drive.")
print("In the UI: select the Anima family, pick the DiT / VAE / text-encoder dropdowns, hit Load.")

### 4️⃣ Import LoRAs / checkpoints from Civitai or HuggingFace

Run this cell **anytime — even mid-session while the UI is serving**: new files
appear in the UI dropdowns after a refresh, no restart needed (the engine scans
the model folders live).

In [ ]:
# @title 4️⃣ Import LoRAs / checkpoints from Civitai or HuggingFace {"form-width":"30%"}
# Paste one URL per line. Civitai links should look like either:
#   https://civitai.com/api/download/models/<versionId>?token=<optional>
#   https://civitai.com/models/<id>?modelVersionId=<versionId>
# Some models require a token: civitai.com → Account settings → API keys.

CIVITAI_TOKEN = ""  # @param {type:"string"}
FILE_URLS = ""      # @param {type:"string"}
DESTINATION = "loras"  # @param ["loras", "diffusion-models", "checkpoints", "vae", "text-encoders", "upscalers", "detailers"]
CUSTOM_FILENAME = ""  # @param {type:"string"}  (optional; only used for a single URL)

import re

assert "fetch" in globals(), "Run cell 3 once first — it defines the downloader."

lines = [l.strip() for l in FILE_URLS.splitlines() if l.strip() and not l.strip().startswith("#")]
assert lines, "Paste at least one download URL above."
if len(lines) > 1 and CUSTOM_FILENAME.strip():
    print("(CUSTOM_FILENAME applies to a single URL \u2014 ignoring it for multiple URLs.)")
    CUSTOM_FILENAME = ""
for url in lines:
    name = CUSTOM_FILENAME.strip() or None
    if "civitai.com" in url:
        if "/api/download/" not in url:
            m = re.search(r"modelVersionId=(\d+)", url)
            if not m:
                raise SystemExit(
                    f"Unsupported Civitai URL: {url}\nUse the download button's link "
                    "(civitai.com/api/download/models/... or ...?modelVersionId=...).")
            url = f"https://civitai.com/api/download/models/{m.group(1)}"
        token = CIVITAI_TOKEN.strip()
        if token and "token=" not in url:
            url += ("&" if "?" in url else "?") + "token=" + token
    fetch(url, DRIVE_MODELS / DESTINATION, name)

print(f"\nDone → {DRIVE_MODELS / DESTINATION}")
print("Refresh the Diffucore UI (F5) — new files appear in the dropdowns, no restart needed.")

### 5️⃣ Launch the UI + ngrok tunnel

In [ ]:
# @title 5️⃣ Launch the UI and open the ngrok tunnel {"form-width":"30%"}

NGROK_TOKEN = ""   # @param {type:"string"}  (or add a Colab Secret named NGROK_TOKEN)
NGROK_DOMAIN = ""  # @param {type:"string"}  (optional: your static ngrok domain)
UI_PASSWORD = ""   # @param {type:"string"}  (recommended: protects the public URL)

import sys, time, socket, subprocess
from pathlib import Path

assert "UI_DIR" in globals(), "Run cells 1 and 2 first."

def port_open(p, host="127.0.0.1"):
    with socket.socket() as s:
        return s.connect_ex((host, p)) == 0

# Restart clean if a previous instance is running.
subprocess.run("pkill -f 'backend/app.py'", shell=True)
time.sleep(2)

LOG = UI_DIR / "server.log"
log = open(LOG, "a")
cmd = [sys.executable, str(UI_DIR / "backend" / "app.py"), "--listen", "--port", str(PORT)]
if UI_PASSWORD.strip():
    cmd += ["--auth-token", UI_PASSWORD.strip()]
proc = subprocess.Popen(cmd, cwd=str(UI_DIR), stdout=log, stderr=subprocess.STDOUT)
print("Starting Diffucore UI (PID", proc.pid, ") ...")

t0 = time.time()
while time.time() - t0 < 240:
    if proc.poll() is not None:
        print("✗ Server exited early. Last log lines:")
        print("\n".join(LOG.read_text(errors="ignore").splitlines()[-40:]))
        raise SystemExit(1)
    if port_open(PORT):
        print("✓ UI ready on port", PORT)
        break
    time.sleep(2)
else:
    print("✗ Timed out waiting for the server. Last log lines:")
    print("\n".join(LOG.read_text(errors="ignore").splitlines()[-40:]))
    raise SystemExit(1)

# ---- ngrok tunnel ----
token = NGROK_TOKEN.strip()
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("NGROK_TOKEN")
    except Exception:
        token = ""
assert token, "Set NGROK_TOKEN above or add a Colab Secret named NGROK_TOKEN (get it at ngrok.com → Your Authtoken)."

from pyngrok import ngrok
ngrok.set_auth_token(token)
ngrok.kill()
kwargs = {}
if NGROK_DOMAIN.strip():
    kwargs["domain"] = NGROK_DOMAIN.strip()
url = ngrok.connect(PORT, "http", **kwargs).public_url

print()
print("=" * 62)
print("🌍  ANIMA / Diffucore UI:", url)
if UI_PASSWORD.strip():
    print("🔑  Login password:", UI_PASSWORD)
print("🖼️  Images auto-save to Drive:", DRIVE_OUTPUTS)
print("ℹ️  First image pays a one-time GPU warmup — be patient.")
print("=" * 62)

### 6️⃣ Server utilities

In [ ]:
# @title 6️⃣ Server status / stop {"form-width":"30%"}

import subprocess

ACTION = "status"  # @param ["status", "stop server"]

if ACTION == "status":
    running = "proc" in globals() and proc.poll() is None
    print("Server:", f"running (PID {proc.pid})" if running else "stopped")
    if running:
        print("\n--- last server log lines ---")
        print("\n".join((UI_DIR / "server.log").read_text(errors="ignore").splitlines()[-15:]))
    try:
        from pyngrok import ngrok
        for t in ngrok.get_tunnels():
            print("Tunnel:", t.public_url)
    except Exception:
        pass
elif ACTION == "stop server":
    subprocess.run("pkill -f 'backend/app.py'", shell=True)
    print("Server stopped. (Run cell 5 again to relaunch.)")

---
## 📋 Cheat sheet

**Loading a model (top bar):** pick **Anima** → choose
`anima-base-v1.0.safetensors` (DiT) + `qwen_image_vae.safetensors` (VAE) +
`qwen_3_06b_base.safetensors` (text encoder) → **Load**.

**Settings that work well with Anima (auto-applied by the UI):**

| | Base model | Turbo (LoRA or DiT) |
|---|---|---|
| Sampler / scheduler | `er_sde` / `flow` (or `dpmpp_2m` / `beta` at 16–20 steps) | `euler` / `flow` |
| Steps | 28–32 | 8–12 |
| CFG | 4.0 | 1.0 |
| Resolution | 832×1216, 1024×1024 | same |

- **Write long, natural-language prompts** — Anima is LLM-conditioned (Qwen3 + T5), tag-style prompts give weak results.
- **LoRAs:** just type them into the prompt, e.g. `a castle in autumn, <lora:my-lora:0.8>`.
- **img2img:** keep strength low (~0.2–0.4) — Anima's flow schedule erases structure fast.
- **Inpaint:** use denoise ~0.9–1.0 for a clean repaint.
- **Face fix:** toggle the **Detailer** (uses the `face_yolov8n.pt` you downloaded).

**Files on your Drive** (`MyDrive/AnimaDiffucore/`):
- `models/diffusion-models/` `models/checkpoints/` `models/loras/` `models/vae/` `models/text-encoders/` `models/detailers/` `models/upscalers/`
- `outputs/YYYY-MM-DD/NNNNN-seed.png` — every generation with full PNG metadata (browsable in the UI gallery too)

**Tips**
- Mid-session model import: cell 4 → paste URL → run → F5 in the UI. The engine picks up new files live.
- ngrok free shows an interstitial "Visit Site" page once per link — click through.
- If the runtime idles out, re-run cells 1, 2 and 5 (models and images are safe in Drive).
- Colab may warn about the notebook not being authored by Google — that's normal for any third-party notebook.